# Small RAG System — Northgate Hostel Handbook

**Goal:** Answer questions using only the text in the uploaded PDF. If the document does not contain the answer, say so rather than guessing.

This notebook uses:
- `pypdf` to extract PDF text
- `sentence-transformers` (`all-MiniLM-L6-v2`) to create chunk and query embeddings
- cosine similarity to retrieve the top 3 chunks
- Gemini for answer generation

**PDF:** Upload `fictional_hostel_handbook.pdf` from the accompanying files. It is a fictional, six-page handbook created specifically for this assignment; its details do not describe a real institution.

> Before submission: run every cell in order, inspect the outputs, and save the notebook with outputs visible. Add your Gemini API key in Colab **Secrets** under the name `GEMINI_API_KEY`; never paste it into a code cell.

In [1]:
%pip -q install pypdf sentence-transformers google-genai


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.7/401.7 kB 6.1 MB/s eta 0:00:00


## 1. Upload the PDF and extract text

In [2]:
from google.colab import files

uploaded = files.upload()  # Select fictional_hostel_handbook.pdf
if not uploaded:
    raise ValueError("No file was uploaded. Upload fictional_hostel_handbook.pdf and rerun this cell.")

PDF_PATH = next(iter(uploaded))
if not PDF_PATH.lower().endswith(".pdf"):
    raise ValueError(f"Expected a PDF file, but received: {PDF_PATH}")
print("Uploaded file:", PDF_PATH)


Saving fictional_hostel_handbook.pdf to fictional_hostel_handbook.pdf
Uploaded file: fictional_hostel_handbook.pdf


In [3]:
from pypdf import PdfReader

reader = PdfReader(PDF_PATH)
if len(reader.pages) == 0:
    raise ValueError("The uploaded PDF has no pages.")
pages_text = [(page.extract_text() or "") for page in reader.pages]
document_text = "\n\n".join(pages_text).strip()
if not document_text:
    raise ValueError("No extractable text was found in the PDF. Use a text-based PDF or run OCR first.")

print("Number of PDF pages:", len(reader.pages))
print("Total extracted characters:", len(document_text))
print("\nFirst 1,000 characters (sanity check):\n")
print(document_text[:1000])


Number of PDF pages: 6
Total extracted characters: 9101

First 1,000 characters (sanity check):

NORTHGATE RESIDENCE — STUDENT HOSTEL
 HANDBOOK
Fictional training document for a Retrieval-Augmented Generation (RAG) assignment
Policy version: 3.2
Effective date: 1 July 2026
Applies to: Residents of Northgate Residence, a fictional college hostel
Issued by: Northgate Residence Office
PURPOSE AND SCOPE
This handbook describes the local rules, dates, fees, processes, and facilities for Northgate Residence.
The figures and rules in this document are fictional and are provided only for coursework practice. They do
not apply to any real college or hostel.
OFFICE CONTACT AND HOURS
The Residence Office is located in Block A, Room A-014. It is open Monday to Friday from 09:00 to 16:30,
and Saturday from 09:00 to 12:00. The office is closed on Sundays and public holidays. Residents should
submit non-urgent forms during office hours. For urgent safety issues outside office hours, residents
should 

## 2. Chunk the text (800 characters, 150 overlap)

In [4]:
def make_chunks(text, chunk_size=800, overlap=150):
    if chunk_size <= 0:
        raise ValueError("chunk_size must be positive")
    if overlap < 0 or overlap >= chunk_size:
        raise ValueError("overlap must be >= 0 and smaller than chunk_size")
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunk_text = text[start:end].strip()
        if chunk_text:
            chunks.append({
                "chunk_id": len(chunks),
                "start_char": start,
                "end_char": end,
                "text": chunk_text
            })
        if end == len(text):
            break
        start = end - overlap
    return chunks

CHUNK_SIZE = 800
OVERLAP = 150
chunks = make_chunks(document_text, CHUNK_SIZE, OVERLAP)
if not chunks:
    raise ValueError("No text chunks were created. Check the PDF text extraction output.")

print("Chunk size:", CHUNK_SIZE)
print("Overlap:", OVERLAP)
print("Number of chunks:", len(chunks))
print("\nExample chunk (full text):\n")
print(f"Chunk ID: {chunks[0]['chunk_id']}")
print(chunks[0]["text"])


Chunk size: 800
Overlap: 150
Number of chunks: 14

Example chunk (full text):

Chunk ID: 0
NORTHGATE RESIDENCE — STUDENT HOSTEL
 HANDBOOK
Fictional training document for a Retrieval-Augmented Generation (RAG) assignment
Policy version: 3.2
Effective date: 1 July 2026
Applies to: Residents of Northgate Residence, a fictional college hostel
Issued by: Northgate Residence Office
PURPOSE AND SCOPE
This handbook describes the local rules, dates, fees, processes, and facilities for Northgate Residence.
The figures and rules in this document are fictional and are provided only for coursework practice. They do
not apply to any real college or hostel.
OFFICE CONTACT AND HOURS
The Residence Office is located in Block A, Room A-014. It is open Monday to Friday from 09:00 to 16:30,
and Saturday from 09:00 to 12:00. The office is closed on Sundays and public holidays. Residents should
submit


## 3. Create embeddings for every chunk

In [5]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
chunk_texts = [chunk["text"] for chunk in chunks]
chunk_embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

if len(chunk_embeddings) != len(chunks):
    raise RuntimeError("Embedding count does not match the number of text chunks.")
print("Embedding matrix shape:", chunk_embeddings.shape)
print("One embedding vector length:", len(chunk_embeddings[0]))
print("Stored chunk records:", len(chunks))


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding matrix shape: (14, 384)
One embedding vector length: 384
Stored chunk records: 14


## 4. Search only — test retrieval before using Gemini

In [6]:
def retrieve(question, chunk_records, embeddings, top_k=3, model=None):
    if not question or not question.strip():
        raise ValueError("Please enter a non-empty question.")
    if not chunk_records or len(embeddings) == 0:
        raise ValueError("There is no indexed document text. Run the PDF extraction and embedding cells first.")
    if len(chunk_records) != len(embeddings):
        raise ValueError("Chunk records and embeddings have different lengths.")
    if top_k < 1:
        raise ValueError("top_k must be at least 1.")
    if model is None:
        model = embedding_model

    query_embedding = model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]
    scores = embeddings @ query_embedding
    best_indices = np.argsort(scores)[::-1][:min(top_k, len(chunk_records))]
    return [
        {
            "chunk_id": int(i),
            "score": float(scores[i]),
            "text": chunk_records[i]["text"]
        }
        for i in best_indices
    ]

retrieval_test_question = "How much is the refundable security deposit?"
retrieved_test = retrieve(retrieval_test_question, chunks, chunk_embeddings, top_k=3)
print("Question:", retrieval_test_question)
for rank, item in enumerate(retrieved_test, start=1):
    print(f"\n--- Rank {rank} | Chunk {item['chunk_id']} | Cosine similarity: {item['score']:.4f} ---")
    print(item["text"])


Question: How much is the refundable security deposit?

--- Rank 1 | Chunk 3 | Cosine similarity: 0.5002 ---
the Residence Office by
15:00 on the same day. Late arrivals are handled by the duty warden only after the office has recorded the
student's request.
SECURITY DEPOSIT
The refundable security deposit is ■4,000 per resident. It must be paid before a room key is issued. The
deposit is returned after checkout, subject to deductions for unpaid charges, missing keys, or damage
beyond ordinary wear. Refund processing normally takes 15 working days after the final room inspection.
ROOM ALLOCATION
Room allocation is made by the Residence Office based on availability and accessibility requirements.
Residents may not exchange keys or occupy an unassigned bed. Requests for a room change must use
Form NR-04 and include a reason. The office aims to respond within three working days, but a request is
not

--- Rank 2 | Chunk 12 | Cosine similarity: 0.4047 ---
SS
For a first non-safety-related m

**Check this output manually:** Is the chunk that states the security deposit amount among the top three? Similarity scores are relative retrieval signals, not probabilities.

## 5. Connect Gemini safely through Colab Secrets

In [12]:
from google.colab import userdata
from google import genai

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise ValueError(
        "Add your key in Colab: left sidebar → Secrets (key icon) → "
        "add GEMINI_API_KEY, enable notebook access, then rerun this cell. "
        "Do not paste the key directly into the notebook."
    )

client = genai.Client(api_key=GEMINI_API_KEY)
GENERATION_MODEL = "gemini-3.8-flash"
print("Gemini client configured. API key was read from Colab Secrets.")


Gemini client configured. API key was read from Colab Secrets.


## 6. Answer with retrieved evidence and show source chunk IDs + scores

In [14]:
def answer_question(question, chunk_records=None, embeddings=None, top_k=3):
    if chunk_records is None:
        chunk_records = chunks

    if embeddings is None:
        embeddings = chunk_embeddings

    retrieved = retrieve(
        question,
        chunk_records,
        embeddings,
        top_k=top_k
    )

    if not retrieved:
        answer = "I cannot find the answer in the provided document."
        print(answer)
        return {
            "question": question,
            "answer": answer,
            "sources": []
        }

    evidence = "\n\n".join(
        f"[Chunk ID: {item['chunk_id']} | "
        f"Similarity: {item['score']:.4f}]\n{item['text']}"
        for item in retrieved
    )

    prompt = f"""You are a strict question-answering assistant.
Answer the question using ONLY the evidence text below.
Do not use outside knowledge or assumptions.
If the evidence does not explicitly contain the answer, reply exactly:
"I cannot find the answer in the provided document."
If the answer is present, answer concisely and preserve relevant numbers and dates.
Do not infer a policy from a related rule.

EVIDENCE:
{evidence}

QUESTION:
{question}

ANSWER:"""

    response = client.models.generate_content(
        model=GENERATION_MODEL,
        contents=prompt
    )

    answer = getattr(response, "text", None)

    if not answer or not answer.strip():
        raise RuntimeError(
            "Gemini returned an empty response. Check the model name, "
            "API key, quota, and connection, then retry."
        )

    answer = answer.strip()
    print(answer)

    print("\nRetrieved sources:")
    for item in retrieved:
        print(
            f"- Chunk {item['chunk_id']} | "
            f"Similarity score: {item['score']:.4f}"
        )

    return {
        "question": question,
        "answer": answer,
        "sources": retrieved
    }


demo_result = answer_question(
    "What is the monthly accommodation fee?"
)


■6,800 per resident (for the 2026–2027 academic year).

Retrieved sources:
- Chunk 4 | Similarity score: 0.6206
- Chunk 12 | Similarity score: 0.4302
- Chunk 13 | Similarity score: 0.3975


## 7. Required six tests

The first three questions have answers in this fictional handbook. The last three are deliberately related to hostel operations but ask for details that the handbook does **not** specify. Inspect all retrieved chunks as well as the generated answer.

## 8. Compulsory experiment — use 3,000-character chunks

In [22]:
LARGE_CHUNK_SIZE = 3000
large_chunks = make_chunks(document_text, chunk_size=LARGE_CHUNK_SIZE, overlap=OVERLAP)
if not large_chunks:
    raise ValueError("No large chunks were created. Check the extracted PDF text.")
large_chunk_texts = [chunk["text"] for chunk in large_chunks]
large_chunk_embeddings = embedding_model.encode(
    large_chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Original chunk size:", CHUNK_SIZE)
print("Original chunk count:", len(chunks))
print("Large chunk size:", LARGE_CHUNK_SIZE)
print("Large chunk count:", len(large_chunks))
print("Overlap kept at:", OVERLAP)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Original chunk size: 800
Original chunk count: 14
Large chunk size: 3000
Large chunk count: 4
Overlap kept at: 150


In [20]:
large_experiment_results = []
for i, question in enumerate(in_document_questions, start=1):
    print("\\n" + "=" * 90)
    print(f"LARGE-CHUNK EXPERIMENT — QUESTION {i} of 3")
    result = answer_question(
        question,
        chunk_records=large_chunks,
        embeddings=large_chunk_embeddings,
        top_k=3
    )
    large_experiment_results.append(result)

\n==========================================================================================
LARGE-CHUNK EXPERIMENT — QUESTION 1 of 3
The refundable security deposit is ■4,000 per resident.

Retrieved sources:
- Chunk 1 | Similarity score: 0.3675
- Chunk 0 | Similarity score: 0.3042
- Chunk 2 | Similarity score: 0.2024
\n==========================================================================================
LARGE-CHUNK EXPERIMENT — QUESTION 2 of 3
The monthly accommodation fee is ■6,800 per resident, and it is due on or before the fifth calendar day of each month.

Retrieved sources:
- Chunk 1 | Similarity score: 0.5996
- Chunk 3 | Similarity score: 0.3551
- Chunk 0 | Similarity score: 0.3468
\n==========================================================================================
LARGE-CHUNK EXPERIMENT — QUESTION 3 of 3
Quiet hours are 22:00–06:00 daily.

Retrieved sources:
- Chunk 3 | Similarity score: 0.3935
- Chunk 2 | Similarity score: 0.2709
- Chunk 0 | Similarity score: 0.